In [1]:
from dotenv import load_dotenv
from google.colab import drive
import os

drive.mount('/content/drive')
load_dotenv('/content/drive/MyDrive/llm-training-pipeline/.env')
os.chdir('/content/drive/MyDrive/llm-training-pipeline')

!git pull

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.


In [ ]:
!pip install -q trl peft bitsandbytes transformers accelerate datasets wandb

In [ ]:
!pip install -q pyarrow --upgrade

In [ ]:
import torch
import wandb
import os
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from dotenv import load_dotenv

In [5]:
from huggingface_hub import login

login(token=os.getenv("HF_TOKEN"))
wandb.login(key=os.getenv("WANDB_API_KEY"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: derek-nguyen99 (derek-nguyen99-self) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
dataset = load_dataset("sahil2801/CodeAlpaca-20k")
print(dataset)
print(f"\nTotal examples: {len(dataset['train'])}")
print(f"\nExample:")
print(dataset['train'][0])

README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json:   0%|          | 0.00/8.06M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['output', 'instruction', 'input'],
        num_rows: 20022
    })
})

Total examples: 20022

Example:
{'output': 'arr = [2, 4, 6, 8, 10]', 'instruction': 'Create an array of length 5 which contains all even numbers between 1 and 10.', 'input': ''}


In [7]:
def format_example(example):
    # If there's additional input context, include it
    if example['input'].strip():
        user_message = f"{example['instruction']}\n\n{example['input']}"
    else:
        user_message = example['instruction']
    
    # Format as a simple instruction/response pair
    text = f"""### Instruction:
{user_message}

### Response:
{example['output']}"""
    return {"text": text}

# Apply formatting
dataset = dataset.map(format_example)

# Inspect a formatted example
print(dataset['train'][0]['text'])
print(f"\nTotal examples: {len(dataset['train'])}")

Map:   0%|          | 0/20022 [00:00<?, ? examples/s]

### Instruction:
Create an array of length 5 which contains all even numbers between 1 and 10.

### Response:
arr = [2, 4, 6, 8, 10]

Total examples: 20022


In [ ]:
# CodeAlpaca only has a train split, so we create our own val set
split = dataset['train'].train_test_split(test_size=0.05, seed=42)
train_dataset = split['train']
val_dataset = split['test']

# Use smaller subset for faster training
train_dataset = train_dataset.select(range(5000))

print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")

print(f"\nSample formatted:")
print(train_dataset[0]['text'])

Train: 19020
Val: 1002

Sample formatted:
### Instruction:
Convert a given number from binary to hexadecimal using C programming language.

10101

### Response:
#include <stdio.h>
#include <math.h>

int main()
{
    int binary=10101;
    int decimal=0, i=0, rem;
    while (binary!=0)
    {
        rem=binary%10;
        decimal=decimal+rem*pow(2,i);
        binary=binary/10;
        ++i;
    }
    printf("The hexadecimal equivalent is : %x", decimal);
    return 0;
}


In [ ]:
model_id = "Qwen/Qwen2.5-0.5B"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Preparing model for kbit training...")
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

print(f"Model loaded ✓")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading tokenizer...


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Preparing model for kbit training...
Model loaded ✓
VRAM used: 1.62 GB


In [10]:
lora_config = LoraConfig(
    r=64,                    # LoRA rank -- number of adapter parameters
    lora_alpha=128,          # scaling factor (usually 2x rank)
    lora_dropout=0.05,       # dropout on adapter layers
    bias="none",             # don't train biases
    task_type="CAUSAL_LM",   # we're doing causal language modeling
    target_modules=[         # which layers to add LoRA adapters to
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention
        "gate_proj", "up_proj", "down_proj",       # MLP
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 73,859,072 || all params: 1,617,573,376 || trainable%: 4.5660


In [19]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, lora_config)

# Force all trainable params to float16
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)

# Verify no bfloat16 trainable params remain
dtypes = set(p.dtype for p in model.parameters() if p.requires_grad)
print(f"Trainable param dtypes: {dtypes}")  # should only show torch.float16
model.print_trainable_parameters()
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Trainable param dtypes: {torch.float16}
trainable params: 73,859,072 || all params: 1,617,573,376 || trainable%: 4.5660
VRAM used: 5.68 GB


In [27]:
sft_config = SFTConfig(
    output_dir="checkpoints/sft-r64",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    logging_steps=50,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=71,
    lr_scheduler_type="cosine",
    fp16=False,
    bf16=False,
    optim="adamw_torch",        # explicit optimizer, no fused issues
    max_length=256,
    dataset_text_field="text",
    report_to="wandb",
    run_name="sft-qwen2.5-1.5b-code-r64",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

In [28]:
wandb.init(
    project="llm-training-pipeline",
    name="sft-qwen2.5-1.5b-code-r64",
    reinit="finish_previous",
    config={
        "model": model_id,
        "dataset": "CodeAlpaca-20k",
        "lora_rank": 64,
        "lora_alpha": 128,
        "epochs": 2,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 16,
        "learning_rate": 2e-4,
        "max_length": 256,
    }
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print("Starting SFT training...")
trainer.train()
print("\nTraining complete!")
wandb.finish()

Starting SFT training...


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [22]:
wandb.finish()

In [14]:
import gc
import torch

print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

VRAM used: 2.27 GB
VRAM free: 13.36 GB
